In [ ]:
import logging
import os
import sys

import warnings
warnings.filterwarnings("ignore", category=UserWarning)
import hydra 
from omegaconf import DictConfig

from config.dinov3_ori.dinov3_l_lap import DINOv3LConfig as base_l
from config.dinov3_ori.dinov3_m_lap import DINOv3MConfig as base_m
from config.dinov3_ori.dinov3_s_lap import DINOv3SConfig as base_s

from hydra.core.config_store import ConfigStore
import time
import datetime
from tqdm import tqdm
import torch
import torch.distributed as dist
import torch.multiprocessing as mp
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.amp import autocast, GradScaler
import torch.optim as optim
from torch.utils.data.distributed import DistributedSampler
from torch.utils.data import DataLoader
from lvis import LVIS, LVISEval
import torch.nn as nn
import importlib
import random

from dataloader.data_utils import (get_pipelines, get_o365_ori_dataset, 
                              get_og_ori_dataset, get_lvis_ori_dataset, train_collate, eval_collate)
from model.backbone.dinov3_adapter import DINOv3STAs
from model.encoder.hybrid_encoder import HybridEncoder
from model.decoder.ovdeim_decoder import OVDEIMDecoder
from model.criterion.ovdeim_criterion import OVDEIMCriterion
from model.ovdeim import OVDEIM
from model.ovdeim_postprocessor import OVDEIMPostProcessor

from dist_tools.dist_utils import get_available_gpus, setup
from optim_tools.ema import ModelEMA
from optim_tools.warmup import WarmupConstantCosine
from optim_tools.utils import get_optim_params, log_param_groups_to_swanlab, count_parameters

In [ ]:
cs = ConfigStore.instance()
cs.store(name="base_l", node=base_l)
cs.store(name="base_m", node=base_m)
cs.store(name="base_s", node=base_s)
hydra.initialize(config_path="config", version_base=None)

In [ ]:
args = hydra.compose(config_name="base_l")

In [ ]:
_, _, test_pipeline = get_pipelines(size=args.data.img_scale, 
                                    num_texts=args.data.num_training_classes, 
                                    blank_text=args.data.blank_text,
                                    mixup_prob=args.data.mixup_prob,
                                    apply_moasic=args.data.apply_moasic,
                                    )

In [ ]:
test_dataset = get_lvis_ori_dataset(data_root=args.data.data_lvis_root, ann_file=args.data.ann_lvis_file, 
                                    pipeline_clean=test_pipeline, 
                                    cache_dir=args.data.cache_file_lvis,
                                    lvis_path=args.data.class_text_lvis_path
                                )

In [ ]:
test_dataloader = DataLoader(test_dataset, batch_size=1, shuffle=False,
                                 collate_fn=eval_collate, num_workers=2)

In [ ]:
lvis_gt = LVIS(args.data.ann_lvis_file)

In [ ]:
device = "cuda"

In [ ]:
backbone = DINOv3STAs(**args.backbone)
encoder = HybridEncoder(**args.encoder)
decoder = OVDEIMDecoder(**args.decoder)    
model = OVDEIM(backbone, encoder, decoder, **args.model).to(device)

In [ ]:
postprocessor = OVDEIMPostProcessor(num_classes=args.data.num_classes, 
                                    num_top_queries=args.decoder.num_queries)

In [ ]:
model_state_dict = torch.load('weights/ovdeim_l.pth', map_location=device)

In [ ]:
# model_state_dict = checkpoint['ema_state_dict']['module']
# torch.save(model_state_dict, 'weights/ovdeim_s.pth')
new_state_dict = {}
for k, v in model_state_dict.items():
    if k.startswith('module.'):
        new_state_dict[k[7:]] = v # remove 'module.' prefix
    else:
        new_state_dict[k] = v
model.load_state_dict(new_state_dict) # Load into the underlying model
model.to(device)

In [ ]:
from collections import defaultdict
from lvis import LVIS, LVISResults, LVISEval

In [ ]:
def eval_fixed_ap(num_enc_queries):
    decoder.num_enc_queries = num_enc_queries
    module = model
    module.eval()
    predictions = []
    postprocessor = OVDEIMPostProcessor(num_classes=args.data.num_classes, 
                                        num_top_queries=300+num_enc_queries)
    with torch.no_grad():
        for batch in test_dataloader:
            data = batch['inputs'].to(device).to(torch.float32)
            targets = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in batch['data_samples'].items()}
            outputs, enc_outputs = module(data, targets)
            outputs['pred_logits'] = torch.cat([outputs['pred_logits'], enc_outputs[0]], dim=1)
            outputs['pred_boxes'] = torch.cat([outputs['pred_boxes'], enc_outputs[1]], dim=1)
            preds = postprocessor(outputs, targets)
            for pred, img_id in zip(preds, targets['img_id']):
                for box, score, label in zip(pred['boxes'], pred['scores'], pred['labels']):
                    predictions.append({
                        'image_id': img_id,
                        'category_id': label.item()+1,
                        'bbox': box.tolist(),
                        'score': score.item()
                    })
    topk = 10000
    by_cat = defaultdict(list)
    for ann in predictions:
        by_cat[ann["category_id"]].append(ann)
    results = []
    missing_dets_cats = set()
    for cat, cat_anns in by_cat.items():
        if len(cat_anns) < topk:
            missing_dets_cats.add(cat)
        results.extend(sorted(cat_anns, key=lambda x: x["score"], reverse=True)[:topk])
    results = LVISResults(lvis_gt, results, max_dets=300+num_enc_queries)
    lvis_eval = LVISEval(lvis_gt, results, iou_type='bbox',)
    params = lvis_eval.params
    params.max_dets = 300+num_enc_queries   # No limit on detections per image.
    lvis_eval.run()
    eval_results = lvis_eval.get_results()
    # lvis_eval.print_results()
    metrics = {
        "AP": eval_results['AP'],
        "AP50": eval_results['AP50'],
        "AP75": eval_results['AP75'],
        "APs": eval_results['APs'],
        "APm": eval_results['APm'],
        "APl": eval_results['APl'],
        "APr": eval_results['APr'],
        "APc": eval_results['APc'],
        "APf": eval_results['APf'],
    }
    print(metrics)

In [ ]:
eval_fixed_ap(num_enc_queries=700)

In [ ]:
def eval_ap():
    decoder.num_enc_queries = 0
    module = model
    module.eval()
    predictions = []
    postprocessor = OVDEIMPostProcessor(num_classes=args.data.num_classes, 
                                        num_top_queries=300)
    with torch.no_grad():
        for batch in test_dataloader:
            data = batch['inputs'].to(device).to(torch.float32)
            targets = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in batch['data_samples'].items()}
            outputs = module(data, targets)
            preds = postprocessor(outputs, targets)
            for pred, img_id in zip(preds, targets['img_id']):
                for box, score, label in zip(pred['boxes'], pred['scores'], pred['labels']):
                    predictions.append({
                        'image_id': img_id,
                        'category_id': label.item()+1,
                        'bbox': box.tolist(),
                        'score': score.item()
                    })
    lvis_eval = LVISEval(lvis_gt, predictions, iou_type='bbox')
    lvis_eval.run()
    eval_results = lvis_eval.get_results()
    lvis_eval.print_results()
    metrics = {
        "AP": eval_results['AP'],
        "AP50": eval_results['AP50'],
        "AP75": eval_results['AP75'],
        "APs": eval_results['APs'],
        "APm": eval_results['APm'],
        "APl": eval_results['APl'],
        "APr": eval_results['APr'],
        "APc": eval_results['APc'],
        "APf": eval_results['APf'],
    }
    print(metrics)   

In [ ]:
eval_ap()

In [ ]:

from dataloader.dataset import MultiModalDataset

lvis_dataset = MultiModalDataset(root=args.data.data_lvis_root, annFile=args.data.ann_lvis_file, detection=True, 
                                lvis=True, lvis_path=args.data.class_text_lvis_path, return_img=False)

In [ ]:
texts = lvis_dataset.class_texts

In [ ]:
decoder.num_enc_queries = 0
module = model
module.eval()
predictions = []
postprocessor = OVDEIMPostProcessor(num_classes=args.data.num_classes, 
                                    num_top_queries=300)
with torch.no_grad():
    for i, batch in enumerate(test_dataloader): # bs = 1
        data = batch['inputs'].to(device).to(torch.float32)
        targets = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in batch['data_samples'].items()}
        outputs = module(data, targets)
        preds = postprocessor(outputs, targets)
        for pred, img_id in zip(preds, targets['img_id']):
            for box, score, label in zip(pred['boxes'], pred['scores'], pred['labels']):
                predictions.append({
                    'image_id': img_id,
                    'category_id': label.item()+1,
                    'bbox': box.tolist(), # x, y, w, h
                    'score': score.item(),
                    'text': texts[label.item()]  # Assuming batch size of 1
                })

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
import os
import colorsys

def visualize_prediction(
    img_id,
    lvis_dataset,
    predictions,
    texts,
    save_path=None,
    figsize=(12, 8),
    dpi=1200,
    score_threshold=0.3,
    max_dets=20,
    label_offset_step=10,  
    fontsize = 9
):
    image_path = lvis_dataset._load_image_path(img_id)
    image = lvis_dataset._load_image(image_path)
    
    preds = [p for p in predictions if p['image_id'] == img_id]
    if not preds:
        print(f"No predictions for image_id {img_id}")
        return

    preds = sorted(preds, key=lambda x: x['score'], reverse=True)
    preds = [p for p in preds if p['score'] >= score_threshold][:max_dets]

    from collections import defaultdict
    bbox_groups = defaultdict(list)
    
    for pred in preds:
        key = tuple(np.round(pred['bbox'], decimals=1))
        bbox_groups[key].append(pred)
    
    fig, ax = plt.subplots(1, figsize=figsize, dpi=dpi)
    ax.imshow(image)
    ax.axis('off')

    all_labels = {p['category_id'] - 1 for p in preds}

    color_map = {}
    for i, label in enumerate(all_labels):
        h = (i / len(all_labels)) % 1.0
        s = 0.9   
        v = 0.95  
        color_map[label] = colorsys.hsv_to_rgb(h, s, v)
    # np.random.seed(46)
    # color_map = {label: np.random.rand(3) for label in all_labels}

    for bbox_key, group_preds in bbox_groups.items():
        x1, y1, w, h = bbox_key
    
        top_pred = max(group_preds, key=lambda p: p['score'])
        label_idx = top_pred['category_id'] - 1
        rect = patches.Rectangle(
            (x1, y1), w, h,
            linewidth=2,
            edgecolor=color_map[label_idx],
            facecolor='none'
        )
        ax.add_patch(rect)

        for i, pred in enumerate(group_preds):
            label_idx = pred['category_id'] - 1
            class_name = texts[label_idx] if label_idx < len(texts) else "unknown"
            score = pred['score']
            label_text = f"{class_name}: {score:.2f}"

            offset_y = y1 - 5 - i * label_offset_step

            ax.text(
                x1, offset_y,
                label_text,
                color='white',
                fontsize=fontsize,
                weight='bold',
                bbox=dict(facecolor='black', alpha=0.7, edgecolor='none', pad=1)
            )

    plt.tight_layout(pad=0)

    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, bbox_inches='tight', pad_inches=0, dpi=dpi, format='png')
        print(f"✅ Saved high-res visualization to: {save_path}")
    else:
        plt.show()

    plt.close()

In [ ]:
ids = lvis_dataset.ids

In [ ]:
import random
for img_id in random.sample(ids, 10):  
    visualize_prediction(
        img_id=img_id,  
        lvis_dataset=lvis_dataset,
        predictions=predictions,
        texts=texts,
        save_path=f"data/visualizations/lvis_{img_id}_predictions.png",
        figsize=(12, 8),
        dpi=300,
        score_threshold=0.5,
        max_dets=300
    )

In [ ]:
from PIL import Image

def resize_and_pad(image_path, target_size=(1024, 1024), fill_value=(255, 255, 255)):
    image = Image.open(image_path).convert("RGB")
    target_w, target_h = target_size
    orig_w, orig_h = image.size
    
    scale = min(target_w / orig_w, target_h / orig_h)
    new_w = int(orig_w * scale)
    new_h = int(orig_h * scale)
    
    resized = image.resize((new_w, new_h), Image.Resampling.LANCZOS)
    new_image = Image.new("RGB", (target_w, target_h), fill_value)
    paste_x = (target_w - new_w) // 2
    paste_y = (target_h - new_h) // 2
    new_image.paste(resized, (paste_x, paste_y))
    
    return new_image

image_path = "data/visualizations/lvis_97337_predictions.png"
resized_img = resize_and_pad(image_path, target_size=(4096, 4096), fill_value=(114, 114, 114))
resized_img.save("data/visualizations/lvis_97337_predictions_resized.png")

In [ ]:
img_id = 97337
visualize_prediction(
        img_id=img_id,  
        lvis_dataset=lvis_dataset,
        predictions=predictions,
        texts=texts,
        save_path=f"data/visualizations/lvis_{img_id}_predictions.png",
        figsize=(12, 8),
        dpi=1200,
        score_threshold=0.5,
        max_dets=300,
        label_offset_step=15,
        fontsize=12
    )

In [ ]:
from dataloader.dataset import MultiModalDataset

In [ ]:
data_lvis_root: str = "/data/Dataset/lvis/"
ann_lvis_file: str = "/data/Dataset/lvis/annotations/lvis_v1_minival_inserted_image_name.json"

In [ ]:
dataset = MultiModalDataset(root=data_lvis_root, annFile=ann_lvis_file, detection=True,
                lvis=True, lvis_path="data/lvis_v1_class_texts.json", return_img=False)

In [ ]:
len(dataset)

In [ ]:
for i in range(len(dataset)):
    path, _ = dataset[i]
    print(path)